# Geo-SR — runnable demo notebook

Structure-Guided, Degradation-Aware Super-Resolution for satellite imagery.

This notebook lets you **run the Geo-SR code end to end**:

1. **Setup & smoke test** — forward/backward on synthetic tensors (needs only `torch`).
2. **Degradation pipeline** — turn an HR tile into a realistic LR one.
3. **Train end-to-end on procedural "urban" tiles** (buildings + roads) — watch the
   loss fall and Geo-SR beat bicubic **without any dataset download** (CPU-friendly).
4. **Point it at your real data** — OLI2MSI / ALSAT-2B via your existing manifest,
   plus a real-degradation evaluation set.
5. **Integrate into your baseline notebook** — the 3 edits, verbatim.

The reference implementation lives in `satellite_sr/geosr/`. Cells 1–3 run anywhere
with PyTorch; `matplotlib` is optional (plots are skipped if it is missing).

In [ ]:
# 1 · Setup ---------------------------------------------------------------
import sys, time, math, random
from pathlib import Path

# Make the `satellite_sr` package importable no matter where the notebook is run.
REPO = Path.cwd().resolve()
for _ in range(5):
    if (REPO / "satellite_sr").is_dir():
        break
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
import torch.nn.functional as F

from satellite_sr.geosr import (
    geosr_base, geosr_tiny, make_geosr_model,
    StructureAwareLoss, SatelliteDegradation,
)
from satellite_sr.geosr.model import GeoSRNet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

print("torch", torch.__version__, "| device:", DEVICE, "| matplotlib:", HAVE_PLT)
print("repo root:", REPO)

## 1 · Smoke test — forward, backward, loss

Confirms the model, the auxiliary edge head, and the composite loss are wired
correctly. `geosr_tiny` is a small config for CPU; `geosr_base` is the config you
would actually train (≈ 8.7M params).

In [ ]:
def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

for name, m in [("geosr_tiny (x3)", geosr_tiny(scale=3)),
                ("geosr_base (x3)", geosr_base(scale=3)),
                ("geosr_base (x4)", geosr_base(scale=4))]:
    print(f"{name:18s}  {n_params(m)/1e6:6.2f} M params  | needs_upsampled={m.needs_upsampled}")

model = geosr_tiny(scale=3).to(DEVICE)
crit  = StructureAwareLoss().to(DEVICE)

lr = torch.rand(2, 3, 24, 24, device=DEVICE)
hr = torch.rand(2, 3, 72, 72, device=DEVICE)
sr, aux = model(lr, return_aux=True)
loss, comps = crit(sr, hr, edge_pred=aux["edge_pred"])
loss.backward()

print("\nSR shape:", tuple(sr.shape), "| range [%.2f, %.2f]" % (float(sr.min().detach()), float(sr.max().detach())))
print("total loss: %.4f" % loss.item())
print("loss terms:", {k: round(v, 4) for k, v in comps.items()})

## 2 · The satellite degradation pipeline

The reason many SR papers report implausible numbers is training/testing on the
*same* bicubic down-sampling. `SatelliteDegradation` synthesises a realistic LR
(sensor blur → resample → shot/read noise → atmosphere → quantisation) so the
model learns a robust inverse. Below we degrade a procedural HR tile.

In [ ]:
# --- procedural "urban" tile generator: rectangles = buildings, lines = roads ---
# Dense, thin high-frequency structure -> exactly the content bicubic blurs away
# and a structure-aware model should recover.
def make_urban_hr(n, size=96, seed=0):
    rng = random.Random(seed)
    out = torch.empty(n, 3, size, size)
    for i in range(n):
        base = torch.tensor([rng.uniform(0.2, 0.45) for _ in range(3)]).view(3, 1, 1)
        img = base + torch.randn(3, size, size) * 0.02          # textured ground
        for _ in range(rng.randint(4, 7)):                      # roads: thin bands (1-2 px)
            w = rng.randint(1, 2); v = rng.uniform(0.1, 0.55)
            if rng.random() < 0.5:
                y = rng.randint(0, size - w); img[:, y:y+w, :] = v
            else:
                x = rng.randint(0, size - w); img[:, :, x:x+w] = v
        for _ in range(rng.randint(12, 20)):                    # many small buildings
            bh, bw = rng.randint(3, 10), rng.randint(3, 10)
            y, x = rng.randint(0, size - bh), rng.randint(0, size - bw)
            col = torch.tensor([rng.uniform(0.4, 0.95) for _ in range(3)]).view(3, 1, 1)
            img[:, y:y+bh, x:x+bw] = col
            img[:, y:y+bh, x:x+1] = 0.97; img[:, y:y+bh, x+bw-1:x+bw] = 0.97
            img[:, y:y+1, x:x+bw] = 0.97; img[:, y+bh-1:y+bh, x:x+bw] = 0.97
        out[i] = img.clamp(0, 1)
    return out

SCALE = 3
degrade = SatelliteDegradation(scale=SCALE)
hr_demo = make_urban_hr(1, size=96, seed=7)
lr_demo = degrade(hr_demo)
bic_demo = F.interpolate(lr_demo, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)
print("HR", tuple(hr_demo.shape), "-> LR", tuple(lr_demo.shape), "(x%d)" % SCALE)

if HAVE_PLT:
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    for a, im, t in zip(ax, [hr_demo, lr_demo, bic_demo],
                        ["HR (ground truth)", "LR (satellite degradation)", "LR -> bicubic x%d" % SCALE]):
        a.imshow(im[0].permute(1, 2, 0).numpy()); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("(install matplotlib to see the images)")

## 3 · Train Geo-SR end-to-end on procedural tiles

No dataset needed. We generate a small set of urban tiles, degrade them to LR, and
train `geosr_tiny` with the composite loss. You should see the **training loss fall**
and **Geo-SR beat bicubic on test PSNR**. This validates the full train/eval loop;
real gains need `geosr_base`, real data, and many epochs (see §4).

> CPU: keep `SIZE=64`, `STEPS≈150` (~1–2 min). GPU: bump both freely.

In [ ]:
def _match(a, b):                       # crop to common H,W (robust to rounding)
    h = min(a.shape[-2], b.shape[-2]); w = min(a.shape[-1], b.shape[-1])
    return a[..., :h, :w], b[..., :h, :w]

def psnr(a, b):
    a, b = _match(a, b)
    mse = ((a - b) ** 2).mean().clamp_min(1e-10)
    return float(10 * torch.log10(1.0 / mse))

# ---- data ----  (HR side kept divisible by SCALE so LR*SCALE == HR)
SIZE       = 22 * SCALE
N_TRAIN    = 80
N_TEST     = 12
BATCH      = 8
STEPS      = 250

hr_train = make_urban_hr(N_TRAIN, size=SIZE, seed=1)
hr_test  = make_urban_hr(N_TEST,  size=SIZE, seed=999)

# A controlled degradation for the CPU demo: STRONG Gaussian blur + downsample +
# light noise. Strong blur is exactly what bicubic cannot undo, so a trained model
# beats bicubic visibly within a couple of minutes. (SatelliteDegradation in
# section 2 is the full randomised pipeline for real robustness training -- see
# section 4, SYNTH_LR=True.)
from satellite_sr.geosr.degradations import apply_blur, apply_resize
def deg(hr):
    x = apply_blur(hr, sigma_range=(1.8, 2.6), aniso=False)
    x = apply_resize(x, SCALE)
    return (x + torch.randn_like(x) * 0.008).clamp(0, 1)

lr_test = torch.stack([deg(hr_test[i:i+1])[0] for i in range(N_TEST)])   # fixed test LR

# ---- model / loss / opt ----
model = geosr_tiny(scale=SCALE).to(DEVICE)
crit  = StructureAwareLoss().to(DEVICE)                 # the full composite loss
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)

@torch.no_grad()
def eval_psnr():
    model.eval()
    sr_p, bic_p = [], []
    for i in range(N_TEST):
        lr = lr_test[i:i+1].to(DEVICE); hr = hr_test[i:i+1].to(DEVICE)
        sr = model(lr)
        bic = F.interpolate(lr, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)
        sr_p.append(psnr(sr, hr)); bic_p.append(psnr(bic, hr))
    model.train()
    return sum(sr_p)/len(sr_p), sum(bic_p)/len(bic_p)

print("before training: Geo-SR %.2f dB | bicubic %.2f dB" % eval_psnr())

t0 = time.time(); hist = []
model.train()
for step in range(1, STEPS + 1):
    idx = torch.randint(0, N_TRAIN, (BATCH,))
    hr = hr_train[idx].to(DEVICE)
    lr = torch.stack([deg(hr[j:j+1])[0] for j in range(BATCH)])   # fresh degradation = augmentation
    sr, aux = model(lr, return_aux=True)
    loss, _ = crit(sr, hr, edge_pred=aux["edge_pred"])
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    hist.append(loss.item())
    if step % 30 == 0 or step == 1:
        sr_db, bic_db = eval_psnr()
        print("step %3d | loss %.4f | Geo-SR %.2f dB | bicubic %.2f dB" % (step, loss.item(), sr_db, bic_db))

sr_db, bic_db = eval_psnr()
print("\ntrained %d steps in %.0fs" % (STEPS, time.time() - t0))
print("FINAL: Geo-SR %.2f dB vs bicubic %.2f dB  (delta %+.2f dB)" % (sr_db, bic_db, sr_db - bic_db))

In [ ]:
# Visualise a test example: LR (bicubic) vs Geo-SR vs HR
if HAVE_PLT:
    model.eval()
    with torch.no_grad():
        lr = lr_test[0:1].to(DEVICE)
        sr = model(lr).cpu()
        bic = F.interpolate(lr, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1).cpu()
    hr = hr_test[0:1]
    fig, ax = plt.subplots(1, 4, figsize=(15, 4))
    imgs = [bic, sr, hr, (sr - hr).abs().mean(1, keepdim=True).repeat(1,3,1,1)]
    ttl  = ["bicubic x%d" % SCALE, "Geo-SR", "HR", "|Geo-SR - HR|"]
    for a, im, t in zip(ax, imgs, ttl):
        a.imshow(im[0].permute(1,2,0).clamp(0,1).numpy()); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(6,3)); plt.plot(hist); plt.xlabel("step"); plt.ylabel("train loss")
    plt.title("Geo-SR training loss"); plt.tight_layout(); plt.show()
else:
    print("(install matplotlib to see the comparison)")

## 4 · Point Geo-SR at your real OLI2MSI / ALSAT-2B data

This mirrors the `PairedSRDataset` in your baseline notebook, and adds the option to
build LR with the **realistic degradation** instead of trusting the stored bicubic LR
— that is your robustness / generalisation evaluation. Set the paths and run.

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

# ----------------- EDIT THESE -----------------
DATA_ROOT    = Path(r"D:/super_resolution")                        # parent of OLI2MSI/ , ALSAT2B/
MANIFEST_CSV = DATA_ROOT / "SR_project_outputs" / "sr_pair_manifest_scene_level_split.csv"
SELECTED     = "OLI2MSI"      # or "ALSAT2B"
SYNTH_LR     = False          # False: use stored LR; True: rebuild LR from HR via SatelliteDegradation
# ----------------------------------------------

def _resolve(p):
    p = Path(str(p)); return p if p.is_absolute() else DATA_ROOT / p

def _load_rgb(path):
    arr = np.asarray(Image.open(_resolve(path)).convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

class ManifestSRDataset(Dataset):
    def __init__(self, df, scale, synth_lr=False):
        self.df = df.reset_index(drop=True); self.scale = scale
        self.deg = SatelliteDegradation(scale) if synth_lr else None
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; hr = _load_rgb(r["hr_path"])
        lr = self.deg(hr.unsqueeze(0))[0] if self.deg else _load_rgb(r["lr_path"])
        return {"lr": lr, "hr": hr, "pair_key": str(r["pair_key"])}

if MANIFEST_CSV.exists():
    man = pd.read_csv(MANIFEST_CSV)
    man = man[man["dataset"] == SELECTED].reset_index(drop=True)
    scale = int(man["scale"].iloc[0])
    tr = ManifestSRDataset(man[man.split == "train"], scale, SYNTH_LR)
    va = ManifestSRDataset(man[man.split == "val"],   scale, False)
    te = ManifestSRDataset(man[man.split == "test"],  scale, False)
    print(f"{SELECTED}: x{scale} | train {len(tr)} | val {len(va)} | test {len(te)} | synth_LR={SYNTH_LR}")
    train_loader = DataLoader(tr, batch_size=8, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(va, batch_size=1, shuffle=False, num_workers=0)
    test_loader  = DataLoader(te, batch_size=1, shuffle=False, num_workers=0)
else:
    print("Manifest not found:", MANIFEST_CSV)
    print("Edit DATA_ROOT / MANIFEST_CSV above. (The demo in section 3 needs no data.)")

In [ ]:
# Full training on real data with the recommended base config.
# Crop to 48x48 LR patches (attention is quadratic in tokens; cropping multiplies data).
def rand_crop(lr, hr, lr_ps, scale):
    _, h, w = lr.shape
    if h <= lr_ps or w <= lr_ps: return lr, hr
    y = random.randint(0, h - lr_ps); x = random.randint(0, w - lr_ps)
    return (lr[:, y:y+lr_ps, x:x+lr_ps],
            hr[:, y*scale:(y+lr_ps)*scale, x*scale:(x+lr_ps)*scale])

def train_geosr_real(epochs=100, lr_ps=48):
    assert MANIFEST_CSV.exists(), "Set DATA_ROOT / MANIFEST_CSV in the previous cell first."
    model = geosr_base(scale=scale).to(DEVICE)
    crit  = StructureAwareLoss().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    best = -1
    for ep in range(1, epochs + 1):
        model.train(); running = 0.0
        for b in train_loader:
            lr_b, hr_b = [], []
            for k in range(b["lr"].shape[0]):
                c_lr, c_hr = rand_crop(b["lr"][k], b["hr"][k], lr_ps, scale)
                lr_b.append(c_lr); hr_b.append(c_hr)
            lr = torch.stack(lr_b).to(DEVICE); hr = torch.stack(hr_b).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                sr, aux = model(lr, return_aux=True)
                loss, _ = crit(sr, hr, edge_pred=aux["edge_pred"])
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); running += loss.item()
        sched.step()
        # quick val PSNR
        model.eval(); vals = []
        with torch.no_grad():
            for b in val_loader:
                lr = b["lr"].to(DEVICE); hr = b["hr"].to(DEVICE)
                sr = model(lr)
                if sr.shape[-2:] != hr.shape[-2:]:
                    sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False).clamp(0,1)
                vals.append(psnr(sr, hr))
        v = sum(vals)/len(vals)
        print(f"epoch {ep:3d} | train {running/len(train_loader):.4f} | val PSNR {v:.2f} dB")
        if v > best:
            best = v; torch.save({"model_state_dict": model.state_dict(), "scale": scale},
                                 REPO / "satellite_sr" / "geosr_best.pth")
    print("best val PSNR: %.2f dB (checkpoint saved)" % best)
    return model

# train_geosr_real(epochs=100)   # <-- uncomment to run on your data

## 5 · Drop Geo-SR into your existing baseline notebook

Three edits — the shared train/eval loop is otherwise untouched.

**a) `make_model` (cell 11):**
```python
from satellite_sr.geosr import make_geosr_model
def make_model(model_name):
    if model_name.startswith("GeoSR"):
        return make_geosr_model(model_name, scale=SCALE, in_ch=3)
    ...  # existing baselines unchanged
```
then add `"GeoSR"` (and `"GeoSR_no_structure"`, `"GeoSR_no_degradation"` for ablations)
to `MODELS_TO_RUN`.

**b) Loss (cell 12, in `train_standard_model`):**
```python
from satellite_sr.geosr import StructureAwareLoss
from satellite_sr.geosr.model import GeoSRNet
criterion = StructureAwareLoss().to(DEVICE)
...
if isinstance(model, GeoSRNet):
    pred, aux = model(inp, return_aux=True)
else:
    pred, aux = model(inp), {"edge_pred": None}
loss, _ = criterion(pred, hr, edge_pred=aux["edge_pred"])
```

**c) Evaluation** — add a **bicubic-only** row, use the **scene-level split**, and add
a **real-degradation** test set (set `SYNTH_LR=True` in section 4). Then report
PSNR/SSIM/UIQI/ERGAS/SAM as now, plus boundary-F1 on a labelled urban set.

See `satellite_sr/docs/ARCHITECTURE_ANALYSIS.md` §1.1 and §5 for why this matters.